# Ylivertainen v2 — Clinical Association Pipeline

End-to-end workflow for finding statistically valid associations between **target outcomes**
and **predictor variables** in a clinical dataset.

This notebook drives six universal modules:

| Module                       | Purpose                                                    |
|------------------------------|------------------------------------------------------------|
| `schema_infer.py`            | Auto-classify each column (continuous, ordinal, …)         |
| `cleaning.py`                | Apply the schema, audit duplicates, derive new columns     |
| `dda.py`                     | Per-column descriptive stats + SVG plots                   |
| `missingness_resolution.py`  | Missing pattern analysis, flags, MICE multiple imputation  |
| `eda.py`                     | Univariate target × predictor screening (FDR-corrected)    |
| `inferential.py`             | Multivariable logistic regression with Rubin pooling       |

**Pipeline order**

```
load → infer schema → clean → DDA → missingness → derive new cols → DDA again
   → EDA screen → MICE impute → multivariable logistic (Rubin pool) → outputs
```

All outputs land under `output/<stage>/{figures,tables}/` as SVG and CSV.


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from schema_infer import infer_schema, print_schema_template, schema_summary, ColSpec
from cleaning import (apply_schema, audit_duplicates,
                      bin_numeric, bin_datetime, make_missing_flag, combine_categories)
from dda import run_dda
from missingness_resolution import (analyze_missingness, add_missing_flags,
                                    mice_impute, simple_impute)
from eda import screen_associations
from inferential import run_inferential

OUTPUT_ROOT = Path("output")


## 1. Load your data

Change the path to point at your Excel/CSV file. The rest of the notebook is dataset-agnostic.


In [2]:
DATA_PATH = "RPE 2020-2025.xlsx"   # or "yourdata.csv"

if str(DATA_PATH).endswith(".csv"):
    df_raw = pd.read_csv(DATA_PATH)
else:
    df_raw = pd.read_excel(DATA_PATH)

print(f"Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head()


Loaded: 1169 rows × 20 columns


,gads,Personas kods,Vecums,PSA līmenis pirms biops,MRI lesions 1 PIRADS,MRI lesions 2 PIRADS,MRI lesions 3 PIRADS,TNM pirms op. (MDK slēdziens),"Riska grupa (zema -1; vidēja-2, augsta-3 )",gleason grade biopsijā,biospsijas veids,RPE grade,Laiks no biopsijas (veikšanas) līdz operācijai,Pvol,upgrade,upstage,downgrade,RPE TNM,PSA bīlvums,R1?
0,2020,280362-12350,57,16.20,5,0,0,T2cN0M0,augsta,2,transrektāla,2,194,41.0,0,0,0,T2cN0M0,0.395122,0
1,2020,010855-11322,64,10.40,0,0,0,T2N1M0,augsta,2,transrektāla,2,169,26.2,0,0,0,T2cN0M0,0.396947,0
2,2020,100474-10701,45,5.55,3,2,0,T2aN0M0,zema,1,transrektāla,1,142,38.0,0,0,0,T2aN0M0,0.146053,0
3,2020,061143-11286,76,7.24,5,3,3,T2cN0M0,augsta,2,transrektāla,2,107,60.0,0,0,0,T2cN0M0,0.120667,0
4,2020,030658-11140,61,7.20,5,0,0,T3aN0M0,augsta,1,transrektāla,1,213,50.7,0,0,0,T2cN0M0,0.142012,0


### 1a. (Optional) Rename columns to clean snake_case

If your source has long Latvian/Russian/free-text column names, rename them here.
Comment this cell out for new datasets where column names are already clean.


In [3]:
# Example for RPE study — edit/remove for other datasets:
# df_raw.columns = ['year', 'pk', 'age', 'preop_PSA', 'lesion_1_MRI_PIRADS', ...]
# df_raw.columns = [c.strip() for c in df_raw.columns]

df_raw.columns = ['year', 'pk', 'age', 'preop_PSA',
       'lesion_1_MRI_PIRADS', 'lesion_2_MRI_PIRADS', 'lesion_3_MRI_PIRADS',
       'preop_TNM_MDK',
       'risk_group', 'biopsy_gleason_grade',
       'biopsy_type', 'RPE_grade',
       'biopsy_to_RPE_days', 'prostate_volume', 'upgrade',
       'upstage', 'downgrade', 'RPE_TNM', 'PSA_density', 'resection_lines_pos']

df = df_raw.reindex([
    'year', 'biopsy_type',
    'lesion_1_MRI_PIRADS', 'lesion_2_MRI_PIRADS', 'lesion_3_MRI_PIRADS',
    'age', 'preop_PSA', 'preop_TNM_MDK',
    'risk_group', 'biopsy_gleason_grade',
    'RPE_grade',
    'biopsy_to_RPE_days', 'prostate_volume', 'upgrade',
    'upstage', 'downgrade', 'RPE_TNM', 'PSA_density', 'resection_lines_pos', 'pk'], axis=1)

df.head(2)

,year,biopsy_type,lesion_1_MRI_PIRADS,lesion_2_MRI_PIRADS,lesion_3_MRI_PIRADS,age,preop_PSA,preop_TNM_MDK,risk_group,biopsy_gleason_grade,RPE_grade,biopsy_to_RPE_days,prostate_volume,upgrade,upstage,downgrade,RPE_TNM,PSA_density,resection_lines_pos,pk
0,2020,transrektāla,5,0,0,57,16.2,T2cN0M0,augsta,2,2,194,41.0,0,0,0,T2cN0M0,0.395122,0,280362-12350
1,2020,transrektāla,0,0,0,64,10.4,T2N1M0,augsta,2,2,169,26.2,0,0,0,T2cN0M0,0.396947,0,010855-11322


## 2. Infer schema and override

The engine auto-classifies every column. Inspect the printed dict, copy it into the
next cell, edit anything wrong (e.g. force `lesion_2_MRI_PIRADS` to `ordinal`,
mark `pk` as `id`, drop a junk column with `kind="skip"`).


In [4]:
schema = infer_schema(df_raw)
schema_summary(schema)


,column,kind,keep,ordered_levels,nulls,note
0,year,ordinal,True,"[2020, 2021, 2022, 2023, 2024, 2025]",None,
1,pk,id,True,None,None,
2,age,continuous,True,None,None,
3,preop_PSA,continuous,True,None,None,
4,lesion_1_MRI_PIRADS,ordinal,True,"[0, 2, 3, 4, 5]",None,
5,lesion_2_MRI_PIRADS,ordinal,True,"[0, 2, 3, 4, 5]",None,
6,lesion_3_MRI_PIRADS,ordinal,True,"[0, 2, 3, 4, 5]",None,
7,preop_TNM_MDK,text,True,None,None,
8,risk_group,nominal,True,None,None,
9,biopsy_gleason_grade,ordinal,True,"[1, 2, 3, 4, 5]",None,


In [5]:
# Print a paste-back-able template; edit it in the next cell.
print_schema_template(schema);


schema_overrides = {
    'year': ColSpec(name='year', kind='ordinal', ordered_levels=[2020, 2021, 2022, 2023, 2024, 2025]),
    'pk': ColSpec(name='pk', kind='id'),
    'age': ColSpec(name='age', kind='continuous'),
    'preop_PSA': ColSpec(name='preop_PSA', kind='continuous'),
    'lesion_1_MRI_PIRADS': ColSpec(name='lesion_1_MRI_PIRADS', kind='ordinal', ordered_levels=[0, 2, 3, 4, 5]),
    'lesion_2_MRI_PIRADS': ColSpec(name='lesion_2_MRI_PIRADS', kind='ordinal', ordered_levels=[0, 2, 3, 4, 5]),
    'lesion_3_MRI_PIRADS': ColSpec(name='lesion_3_MRI_PIRADS', kind='ordinal', ordered_levels=[0, 2, 3, 4, 5]),
    'preop_TNM_MDK': ColSpec(name='preop_TNM_MDK', kind='text'),
    'risk_group': ColSpec(name='risk_group', kind='nominal'),
    'biopsy_gleason_grade': ColSpec(name='biopsy_gleason_grade', kind='ordinal', ordered_levels=[1, 2, 3, 4, 5]),
    'biopsy_type': ColSpec(name='biopsy_type', kind='nominal'),
    'RPE_grade': ColSpec(name='RPE_grade', kind='ordinal', ordered_levels=[1, 2,

### 2a. Paste the edited schema below

Take the printout from the cell above, paste it here, and adjust kinds/ordered_levels
as needed. Anything you don't override stays as inferred.

For RPE specifically, things to check:
- `pk` → `id`
- `preop_TNM_MDK`, `RPE_TNM` → `nominal`
- `risk_group` → `ordinal` with `ordered_levels=['zema','vidēja','augsta']`
- `biopsy_gleason_grade`, `RPE_grade`, PIRADS columns → `ordinal` with `[1,2,3,4,5]` (or `[0,1,2,3,4,5]`)
- `upgrade`, `upstage`, `downgrade`, `resection_lines_pos` → `binary`


In [6]:
# Example override — adapt to your dataset!
schema_overrides = {
    'year': ColSpec(name='year', kind='ordinal', ordered_levels=[2020, 2021, 2022, 2023, 2024, 2025]),
    'pk': ColSpec(name='pk', kind='id'),
    'age': ColSpec(name='age', kind='continuous', note="create bins"),
    'preop_PSA': ColSpec(name='preop_PSA', kind='continuous'),
    'lesion_1_MRI_PIRADS': ColSpec(name='lesion_1_MRI_PIRADS', kind='ordinal', ordered_levels=[0, 2, 3, 4, 5], nulls=(0,), note="delete NaN rows"),
    'lesion_2_MRI_PIRADS': ColSpec(name='lesion_2_MRI_PIRADS', kind='ordinal', ordered_levels=[0, 2, 3, 4, 5]),
    'lesion_3_MRI_PIRADS': ColSpec(name='lesion_3_MRI_PIRADS', kind='ordinal', ordered_levels=[0, 2, 3, 4, 5]),
    'preop_TNM_MDK': ColSpec(name='preop_TNM_MDK', kind='nominal'),
    'risk_group': ColSpec(name='risk_group', kind='ordinal', ordered_levels=['zema', 'vidēja', 'augsta']),
    'biopsy_gleason_grade': ColSpec(name='biopsy_gleason_grade', kind='ordinal', ordered_levels=[1, 2, 3, 4, 5]),
    'biopsy_type': ColSpec(name='biopsy_type', kind='nominal'),
    'RPE_grade': ColSpec(name='RPE_grade', kind='ordinal', ordered_levels=[1, 2, 3, 4, 5]),
    'biopsy_to_RPE_days': ColSpec(name='biopsy_to_RPE_days', kind='continuous', note="create bins"),
    'prostate_volume': ColSpec(name='prostate_volume', kind='continuous'),
    'upgrade': ColSpec(name='upgrade', kind='binary'),
    'upstage': ColSpec(name='upstage', kind='binary'),
    'downgrade': ColSpec(name='downgrade', kind='binary'),
    'RPE_TNM': ColSpec(name='RPE_TNM', kind='nominal'),
    'PSA_density': ColSpec(name='PSA_density', kind='continuous'),
    'resection_lines_pos': ColSpec(name='resection_lines_pos', kind='binary'),
}
# merge overrides on top of inferred schema:
schema.update(schema_overrides)
schema_summary(schema)


,column,kind,keep,ordered_levels,nulls,note
0,year,ordinal,True,"[2020, 2021, 2022, 2023, 2024, 2025]",None,
1,pk,id,True,None,None,
2,age,continuous,True,None,None,create bins
3,preop_PSA,continuous,True,None,None,
4,lesion_1_MRI_PIRADS,ordinal,True,"[0, 2, 3, 4, 5]",[0],delete NaN rows
5,lesion_2_MRI_PIRADS,ordinal,True,"[0, 2, 3, 4, 5]",None,
6,lesion_3_MRI_PIRADS,ordinal,True,"[0, 2, 3, 4, 5]",None,
7,preop_TNM_MDK,nominal,True,None,None,
8,risk_group,ordinal,True,"[zema, vidēja, augsta]",None,
9,biopsy_gleason_grade,ordinal,True,"[1, 2, 3, 4, 5]",None,


## 3. Apply schema → coerce dtypes, replacements, nulls

This is the only place dtypes are set. Downstream stages trust the schema.


In [7]:
df = apply_schema(df_raw, schema)
df.dtypes


year                    category
pk                        string
age                        int64
preop_PSA                float64
lesion_1_MRI_PIRADS     category
lesion_2_MRI_PIRADS     category
lesion_3_MRI_PIRADS     category
preop_TNM_MDK           category
risk_group              category
biopsy_gleason_grade    category
biopsy_type             category
RPE_grade               category
biopsy_to_RPE_days         int64
prostate_volume          float64
upgrade                  boolean
upstage                  boolean
downgrade                boolean
RPE_TNM                 category
PSA_density              float64
resection_lines_pos      boolean
dtype: object

## 4. Duplicate audit

Provide ID columns. The audit returns rows in duplicate groups and a cleaned frame.


In [8]:
ID_COLS = ['year', 'pk']   # edit for your dataset

dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
print(f"Found {len(dupes)} duplicated rows (across {len(dupes)//2 if len(dupes) else 0}+ groups)")
dupes.to_csv("dupes.csv")


Found 2 duplicated rows (across 1+ groups)


## 5. DDA — first pass

Descriptive stats + SVG plots for every kept column.
Outputs land in `output/dda/`.


In [9]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT, skip_cols=[])

print("\n--- continuous ---");  display(dda_tables['continuous'])
print("\n--- categorical ---"); display(dda_tables['categorical'])
print("\n--- binary ---");      display(dda_tables['binary'])
print("\n--- datetime ---");    display(dda_tables['datetime'])



--- continuous ---


,column,kind,n,missing_pct,mean,sd,median,q1,q3,min,max,skew
0,age,continuous,1169,0.0,64.792130,6.512139,65.00000,60.000000,69.000000,45.000000,84.000000,-0.316532
1,preop_PSA,continuous,1169,0.0,10.879369,8.905501,8.07000,6.000000,12.400000,0.313000,100.000000,3.517718
2,biopsy_to_RPE_days,continuous,1169,0.0,174.175364,233.910131,119.00000,90.000000,166.000000,0.000000,3326.000000,7.096658
3,prostate_volume,continuous,1169,0.0,43.964808,20.905236,39.00000,30.000000,52.000000,2.000000,178.000000,1.703525
4,PSA_density,continuous,1169,0.0,0.294012,0.305423,0.20825,0.138317,0.335714,0.020579,4.545455,5.824991



--- categorical ---


,column,kind,n,missing_pct,n_levels,mode,top3,ordered
0,year,ordinal,1169,0.00,6,2022,"2022=228, 2023=212, 2020=207",True
1,lesion_1_MRI_PIRADS,ordinal,1136,2.82,5,5,"5=526, 4=498, 3=87",True
2,lesion_2_MRI_PIRADS,ordinal,1169,0.00,5,0,"0=881, 4=174, 3=72",True
3,lesion_3_MRI_PIRADS,ordinal,1169,0.00,5,0,"0=1134, 3=16, 4=13",True
4,preop_TNM_MDK,nominal,1169,0.00,17,T2cN0M0,"T2cN0M0=401, T2N0M0=272, T1cN0M0=121",False
5,risk_group,ordinal,1169,0.00,3,vidēja,"vidēja=531, augsta=420, zema=218",True
6,biopsy_gleason_grade,ordinal,1169,0.00,5,2,"2=498, 1=435, 3=134",True
7,biopsy_type,nominal,1169,0.00,2,transrektāla,"transrektāla=748, transperineāla=421",False
8,RPE_grade,ordinal,1169,0.00,5,2,"2=697, 1=184, 3=184",True
9,RPE_TNM,nominal,1169,0.00,25,T3aN0M0,"T3aN0M0=365, T2cN0M0=282, T2N0M0=263",False



--- binary ---


,column,kind,n,missing_pct,p_true,n_true,n_false
0,upgrade,binary,1169,0.0,0.3550,415,754
1,upstage,binary,1169,0.0,0.4457,521,648
2,downgrade,binary,1169,0.0,0.1215,142,1027
3,resection_lines_pos,binary,1169,0.0,0.0659,77,1092



--- datetime ---


""


## 6. Missingness analysis

Per-column %, plus a Jaccard co-missingness heatmap so you can spot blocks of
columns that are missing together (often a data-entry-process artifact).


In [10]:
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary


,column,n_missing,pct_missing
0,lesion_1_MRI_PIRADS,33,2.82
1,year,0,0.00
2,RPE_grade,0,0.00
3,PSA_density,0,0.00
4,RPE_TNM,0,0.00
5,downgrade,0,0.00
6,upstage,0,0.00
7,upgrade,0,0.00
8,prostate_volume,0,0.00
9,biopsy_to_RPE_days,0,0.00


### 6a. Add missingness flags

For variables you suspect are **MNAR** (missing not at random — e.g. PSA missing
because it wasn't measured in low-risk patients), add an explicit flag so
downstream models can use "was-it-measured" as a predictor.


In [11]:
MNAR_COLS = []   # e.g. ['preop_PSA', 'PSA_density']
df = add_missing_flags(df, MNAR_COLS)

# The new flag columns need to be in the schema. They're booleans:
for c in MNAR_COLS:
    flag = f"{c}_missing"
    if flag in df.columns and flag not in schema:
        schema[flag] = ColSpec(name=flag, kind='binary')

df.filter(like='_missing').head()


""
0
1
2
3
4


## 7. Derive new columns (age bins, time bins, PSA categories…)

Use the helpers below freely. Any new column you add **must** also be added to the
schema so DDA/EDA/inferential will analyze it.


In [ ]:
# Examples — edit/delete for your dataset:

# Age bins
if 'age' in df.columns:
    df['age_bin'] = bin_numeric(df['age'], bins=[0, 55, 65, 75, 200],
                                labels=['<55', '55-64', '65-74', '75+'])
    schema['age_bin'] = ColSpec(name='age_bin', kind='ordinal',
                                ordered_levels=['<55', '55-64', '65-74', '75+'])

# PSA categories (clinical cutpoints)
if 'preop_PSA' in df.columns:
    df['psa_cat'] = bin_numeric(df['preop_PSA'], bins=[0, 4, 10, 20, 1e6],
                                labels=['<4', '4-10', '10-20', '>20'])
    schema['psa_cat'] = ColSpec(name='psa_cat', kind='ordinal',
                                ordered_levels=['<4', '4-10', '10-20', '>20'])

# Time-to-surgery bins (days)
if 'biopsy_to_RPE_days' in df.columns:
    df['biopsy_to_rpe_bin'] = bin_numeric(
        df['biopsy_to_RPE_days'], bins=[0, 30, 90, 180, 1e5],
        labels=['≤30d', '31-90d', '91-180d', '>180d'])
    schema['biopsy_to_rpe_bin'] = ColSpec(name='biopsy_to_rpe_bin', kind='ordinal',
                                          ordered_levels=['≤30d', '31-90d', '91-180d', '>180d'])

df.filter(regex='_bin$|^psa_cat$').head()


## 8. DDA — second pass (with derived columns)

Re-run DDA so the new columns get their own plots and stats.


In [ ]:
dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)
dda_tables['categorical']


## 9. Configure targets and predictors

This is the only place outcome variables and candidate predictors are declared.


In [ ]:
TARGETS = ['upgrade', 'upstage', 'downgrade']

# Optional whitelist — leave None to use every testable column except the targets.
PREDICTORS = [
    'biopsy_type', 'age', 'preop_PSA', 'risk_group',
    'biopsy_gleason_grade', 'biopsy_to_RPE_days', 'PSA_density',
    'lesion_1_MRI_PIRADS', 'lesion_2_MRI_PIRADS', 'lesion_3_MRI_PIRADS',
    'age_bin', 'psa_cat', 'biopsy_to_rpe_bin',
]
PREDICTORS = [c for c in PREDICTORS if c in df.columns]  # drop any missing names

# Which value of each target counts as "the event" (positive class).
POSITIVE_CLASS = {t: True for t in TARGETS}


## 10. EDA — univariate screening

For each (target × predictor) pair the engine picks the right test:

| predictor kind | test                     | effect size           |
|----------------|--------------------------|-----------------------|
| continuous     | Mann–Whitney U           | rank-biserial r       |
| ordinal        | Spearman ρ               | ρ                     |
| nominal        | χ² (Fisher if any E<5)   | Cramér's V            |
| binary         | Fisher exact (2×2)       | odds ratio + V        |
| datetime       | MWU on days-since-min    | rank-biserial r       |

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [ ]:
assoc = screen_associations(
    df, schema,
    targets=TARGETS,
    predictors=PREDICTORS,
    positive_class=POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

assoc[assoc.fdr_significant]


In [ ]:
# Full table
assoc


## 11. Multiple imputation (MICE)

We generate **m=10** imputed datasets via sklearn's IterativeImputer
(RandomForest estimator, separate random seed per imputation).
The pooled inferential stage applies Rubin's rules over these 10 fits.

For a quick screening run set `m=3`. For publication use `m≥10`.


In [ ]:
M = 10  # number of imputations; reduce to 3 for fast iteration

imputed_frames = mice_impute(df, schema, m=M, max_iter=10,
                             random_state=42, output_root=OUTPUT_ROOT)
print(f"Generated {len(imputed_frames)} imputed frames")
print("NaN count in first imputed frame:", imputed_frames[0].isna().sum().sum())


## 12. Multivariable logistic regression (Rubin-pooled)

For each target:

1. Build design matrix (continuous z-scored, ordinal kept as codes, nominal one-hot).
2. Iteratively drop predictors with **VIF > 5** to handle collinearity.
3. Fit logistic regression on each of the m imputed frames.
4. Pool coefficients with **Rubin's rules** (Barnard–Rubin df).
5. Report adjusted OR with 95% CI and pooled p-value.
6. Save a forest plot SVG per target.


In [ ]:
inf_results = run_inferential(
    imputed_frames, schema,
    targets=TARGETS,
    predictors=PREDICTORS,
    positive_class=POSITIVE_CLASS,
    vif_threshold=5.0,
    output_root=OUTPUT_ROOT,
)
inf_results


In [ ]:
# Significant adjusted predictors per target
inf_results[(inf_results['p'] < 0.05) & inf_results['or'].notna()]


## 13. Outputs

Everything is saved on disk:

```
output/
├── dda/{figures,tables}/
├── missingness/{figures,tables}/
├── eda/{figures,tables}/
└── inferential/{figures,tables}/
```

Each plot is an individual `.svg`; each result table an individual `.csv`.


In [ ]:
from pathlib import Path
for p in sorted(Path(OUTPUT_ROOT).rglob('*')):
    if p.is_file():
        print(p)


## 14. NOTES — Why each statistical choice

Concise but detailed rationale for every formula used in this pipeline.
For each: **what it does**, **why chosen**, **what was rejected**.

---

### Schema inference (hybrid auto + override)

- **What.** Heuristic classification of each column into `continuous / count / ordinal / nominal / binary / datetime / id / text / skip` based on dtype, cardinality, value patterns.
- **Why.** Test selection downstream is kind-driven — a wrong kind silently picks the wrong test (e.g. treating Gleason 1–5 as `continuous` instead of `ordinal` swaps Spearman for MWU and loses interpretability of "per-grade increase").
- **Alternatives rejected.**
  - *Full auto-only*: brittle on clinical data where 0/1-coded ordinals look numeric.
  - *Manual ColSpec per column*: correct but tedious; you'd re-type 30+ specs per study.

---

### Duplicate auditing on normalized string keys

- **What.** Lowercase + strip + empty→NA on ID columns, then flag rows whose full key tuple is non-null and repeated.
- **Why.** Clinical IDs (`pk`, `year`) frequently have invisible whitespace or case drift across data-entry sessions. Naive `duplicated()` misses these.
- **Alternatives rejected.**
  - *Exact match*: under-detects.
  - *Fuzzy match (Levenshtein)*: over-detects, would falsely merge genuinely different patients.

---

### Mann–Whitney U for continuous/count vs binary outcome

- **What.** Non-parametric rank-sum test. H₀: P(X₁ > X₂) = ½. Two-sided.
- **Effect size.** Rank-biserial **r = |Z|/√N**, where Z is the large-sample normal approximation of U. Bounded 0–1, interpretable like Cohen's r (0.1 small, 0.3 medium, 0.5 large).
- **Why.**
  - Clinical continuous variables (PSA, age, days-to-surgery) are **almost never normal** — PSA in particular is heavily right-skewed.
  - MWU has ~95% efficiency vs t-test under normality and is far more robust under non-normality.
  - One test for the whole pipeline = no test-switching artifacts.
- **Alternatives rejected.**
  - *Welch's t-test always*: violates assumption on skewed data; inflates type-I error on small skewed samples.
  - *Auto Shapiro-Wilk switch (t if normal, MWU else)*: the normality test itself adds noise and its decision is sample-size dependent (always rejects normal at large N, never at small N) — produces worse calibration than just using MWU.
  - *Welch's t on log-transformed data*: works for PSA specifically but not generalizable to all continuous predictors in the pipeline.
- **Sensitivity.** When publishing, re-run Welch's t on log(PSA) as a sensitivity analysis — if direction and significance agree with MWU, you're robust.

---

### Spearman ρ for ordinal vs binary outcome

- **What.** Pearson correlation on the ranks of category codes vs the 0/1-encoded outcome.
- **Why.**
  - Preserves the **ordering** of ordinal predictors (Gleason 1<2<3<4<5, PIRADS 1<2<3<4<5, risk_group low<mid<high). χ² throws this away — it would only tell you "the distribution differs across levels", not "higher Gleason → more upgrades".
  - Yields a signed, scale-free effect size (ρ) that's directly publishable.
- **Alternatives rejected.**
  - *χ² on the ordinal × binary table*: ignores ordering, weaker power, no direction.
  - *Cochran-Armitage trend test*: equivalent to a linear-trend variant of χ² and gives p only — Spearman gives p **plus** a comparable ρ across all ordinal predictors.
  - *Kendall's τ*: similar info but slower on large N and no power advantage here.

---

### χ² (or Fisher exact) for nominal vs binary

- **What.** χ² of independence on the contingency table, **without Yates correction** (modern recommendation — Yates is overconservative). Switches to **Fisher exact** if the 2×2 table has any expected cell count < 5.
- **Why Fisher when expected<5.** χ²'s asymptotic distribution breaks down with small expected counts; Fisher's exact test conditions on the marginals and computes the exact hypergeometric p — correct at any sample size.
- **Effect size: Cramér's V** = √(χ²/(N·(min(r,c)−1))). Bounded 0–1, comparable across table shapes. For 2×2 tables we **also** report the odds ratio because clinicians read OR natively.
- **Alternatives rejected.**
  - *Yates-corrected χ²*: too conservative for modern computing — Fisher is exact and almost as fast.
  - *G-test (likelihood ratio)*: theoretically nicer for nested models but identical conclusions in 2-way tables; less familiar to clinical readers.
  - *Permutation χ²*: same answer as Fisher for 2×2, more expensive.

---

### Benjamini–Hochberg FDR correction, per target

- **What.** Sort p-values ascending; for rank i out of m, compute q_i = p_(i)·m/i; enforce monotonicity from the right; significance at q < α controls expected proportion of false discoveries at α.
- **Why per-target (not pooled across all targets).** Each outcome (upgrade, upstage, downgrade) is a **separate family** of hypotheses with its own scientific interpretation. Pooling them inflates the family size and over-corrects. This matches how clinical journals report multi-outcome studies.
- **Alternatives rejected.**
  - *Bonferroni*: controls family-wise error rate — far too conservative for a screening stage with 10+ predictors. Misses real signal.
  - *Holm-Bonferroni*: still FWER, marginally less conservative than Bonferroni but still much stricter than BH.
  - *Storey q-value*: estimates the null proportion adaptively; great when you have hundreds of tests but unstable at small m (you'll have <20 tests per target).
  - *No correction*: indefensible with ≥3 predictors per target — false discovery rate would be ~30%+.
- **Verified.** Output matches `statsmodels.stats.multitest.multipletests(method='fdr_bh')` exactly.

---

### MICE (Multiple Imputation by Chained Equations), m=10

- **What.** For each missing value: fit a regression of that column on all others using observed data, predict missing values, iterate until convergence. Repeat with m different random seeds to produce m plausible completed datasets.
- **Why multiple (not single).** Single imputation pretends the imputed values are known, so it **understates standard errors**. With m=10 imputations and Rubin pooling, the SEs honestly include imputation uncertainty.
- **Estimator: RandomForestRegressor.** Captures non-linear relationships (PSA × age × Gleason interactions) without you specifying them. Tolerates mixed numeric/categorical inputs.
- **Why m=10.** Rubin showed efficiency = (1 + fmi/m)^(-1) where fmi is fraction of missing info. At fmi ≈ 0.3 (typical clinical data), m=10 gives ~97% efficiency. m=5 is acceptable, m=20 is overkill.
- **Alternatives rejected.**
  - *Mean/median imputation*: distorts variance and any correlation involving the imputed column. Catastrophic for inferential SEs.
  - *Complete-case analysis*: throws away rows with any missingness — typically 20–50% data loss in clinical cohorts; introduces selection bias if missingness is MAR (which it usually is).
  - *Hot-deck imputation*: works for nominal-only data; weaker for mixed types.
  - *Bayesian model-based imputation (`mice` R package, Stan)*: gold standard but heavy infrastructure; sklearn's `IterativeImputer` is close enough for clinical screening.
- **Limitation.** Assumes data is **Missing At Random** (MAR) — missingness depends only on observed variables. For **MNAR** patterns (e.g. "PSA was missing because risk was low"), add explicit `<col>_missing` flags in section 6a so the model can use the missingness indicator itself as a predictor.

---

### Missingness flags

- **What.** Binary indicator columns `<col>_missing` added before imputation.
- **Why.** In clinical data, *that a value was missing* is often informative (e.g. PSA not measured because clinician judged it unnecessary). Including the flag in the regression lets the model separate "the value's effect" from "the act of measuring's effect".
- **Alternatives rejected.**
  - *Imputing without flags*: hides the MNAR mechanism.
  - *Dropping columns with high missingness*: throws away signal; missingness % is not a reliable filter for clinical utility.

---

### Variance Inflation Factor (VIF) pruning, threshold = 5

- **What.** For each predictor x_j, VIF = 1/(1 − R²_j), where R²_j is from regressing x_j on all other predictors. Iteratively drop the column with the highest VIF until all ≤ 5.
- **Why.** Logistic regression with collinear predictors produces enormous standard errors and unstable coefficients ("model can't tell whether PSA or PSA-density is doing the work"). VIF > 5 ⇔ R²_j > 0.80 ⇔ severe multicollinearity.
- **Why threshold = 5** (not 10). VIF=10 is the classical statistics teaching threshold but for clinical regression with modest N (<500), 5 is the modern recommendation (Vatcheva 2016, O'Brien 2007).
- **Alternatives rejected.**
  - *Pairwise Pearson correlation > 0.8*: catches only 2-variable collinearity; misses 3-way (e.g. a = b + c).
  - *Lasso regularization*: would drop collinear features automatically but **biases coefficients** toward zero — bad for inference (you want unbiased OR estimates). Lasso is for prediction, not inference.
  - *Ridge / Elastic Net*: same problem — shrinks coefficients, distorts ORs.
  - *PCA / partial-least-squares*: components are uninterpretable clinically.

---

### Multivariable logistic regression

- **What.** Per target, one binary logistic model with all surviving predictors. Continuous z-scored (so OR is per-SD increase), ordinals kept as numeric codes, nominals one-hot with drop_first.
- **Why.** Univariate screening (section 10) ignores confounding — Gleason can show up "significant" purely because it correlates with PSA. Multivariable estimates the **adjusted** effect of each predictor holding the others constant.
- **Why this design encoding.**
  - *z-score continuous*: ORs comparable across predictors; one "unit" = one SD.
  - *Ordinal as numeric code*: assumes linear log-odds across levels (parsimonious; standard for Gleason/PIRADS in urology papers). The alternative is one-hot with drop_first, which uses more degrees of freedom and is only worth it if the trend is clearly non-monotonic — check the EDA plots first.
  - *Nominal one-hot drop_first*: avoids the dummy variable trap (perfect collinearity with intercept).
- **Alternatives rejected.**
  - *Univariate-only pipeline*: misleading because of confounding.
  - *Stepwise selection (forward/backward)*: notorious for unstable selection, inflated significance, and irreproducibility. Modern guidance (Harrell, Steyerberg) is: don't.
  - *Random forest / XGBoost*: better predictive accuracy but no clinical OR with CI to report.
  - *Penalized regression (Firth, Lasso, Ridge)*: useful with extreme separation or n<<p but biases the OR estimates — defeats the inferential purpose.
  - *Bayesian logistic with weakly informative priors*: cleaner for tiny samples and would give credible intervals — but you'd need to defend prior choice in the manuscript.

---

### Rubin's rules with Barnard–Rubin degrees of freedom

- **What.** Across the m=10 imputed-frame fits, for each coefficient:
  - θ̄ = mean of the m point estimates
  - within-imp variance Ū = mean of the m squared SEs
  - between-imp variance B = sample variance of the m estimates
  - total variance T = Ū + (1 + 1/m)·B
  - pooled SE = √T
  - degrees of freedom (Barnard–Rubin):
    df = (m−1)·(1 + Ū/((1+1/m)·B))²
  - p-value from t-distribution with that df; 95% CI = θ̄ ± t_{0.975, df}·SE
- **Why.** Rubin's rules are the **only** statistically valid way to combine results across multiple imputations. The total variance T splits into "within" (each model's uncertainty) and "between" (uncertainty due to missing data) — they're not interchangeable.
- **Why Barnard–Rubin df (not the original Rubin 1987 df).** Original Rubin df → ∞ when between-variance is small, which is wrong when m is small. Barnard–Rubin (1999) is a small-sample correction that's now the standard (R `mice` uses it, SAS PROC MIANALYZE uses it).
- **Alternatives rejected.**
  - *Picking the "best" imputation*: defeats the purpose of multiple imputation entirely.
  - *Average the imputed datasets first, then fit once*: produces correct point estimates but **wrong SEs** (the between-variance is invisible).
  - *Use the within-variance only*: ignores imputation uncertainty — false confidence.
- **Verified.** With zero between-variance, our pooler returns SE equal to the single-fit SE; with non-zero between, it correctly inflates SE and produces a finite small-sample df (e.g. m=5, modest B → df ≈ 22).

---

### Why log-scale x-axis on forest plots

- ORs are multiplicative (OR=2 and OR=0.5 are equal-and-opposite effects). On a linear axis they look asymmetric; on log scale they're symmetric around OR=1, which is the correct visual.

---

### What this pipeline deliberately does NOT do

- **No machine-learning prediction** (no train/test split, no AUC, no calibration). This is an **association/inference** pipeline, not a prediction pipeline. If you later want a predictive model (e.g. nomogram for upgrade risk), that's a separate workflow with cross-validation, calibration plots, decision-curve analysis.
- **No causal inference** (no DAGs, no IPTW, no instrumental variables). All effects here are **statistical associations** adjusted for the included covariates — they are *not* causal effects. Manuscript wording must say "associated with", never "causes".
- **No survival/time-to-event analysis.** Targets here are binary (upgrade yes/no). If you later care about *time to biochemical recurrence*, you'd need Cox regression — a separate module.

---

### Sanity-check checklist before submitting results

1. Print `schema_summary(schema)` — every ordinal has correct `ordered_levels`?
2. After MICE, `imputed_frames[0].isna().sum().sum()` == 0 for predictor columns?
3. `inf_results['n_models']` ≈ m for all predictors (means the model converged on every imputation)?
4. Forest plot ORs and EDA univariate effects agree in **direction** (sign)? If they flip, you have confounding worth discussing.
5. For each FDR-significant univariate result, check the corresponding plot in `output/eda/figures/` — is the pattern visually credible or driven by 2–3 outliers?
